# Efficient Multi-Modal Product Pricing Model with CLIP

Based on the ML Challenge 2025 - Smart Product Pricing Challenge

## Segment 1: Environment Setup and Data Loading

**Explanation:** This installs all necessary libraries including Transformers for CLIP, LightGBM for efficient gradient boosting, and sentence-transformers for additional text encoding capabilities.

In [1]:
# Install required packages
!pip install torch torchvision transformers pillow tqdm lightgbm scikit-learn pandas numpy sentence-transformers --quiet

In [2]:
import pandas as pd
import numpy as np
import torch 
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import requests
from io import BytesIO
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Load datasets
train_df = pd.read_csv("dataset/train.csv")
test_df = pd.read_csv("dataset/test.csv")

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"\nPrice statistics:")
print(train_df['price'].describe())

ImportError: cannot import name 'CLIPProcessor' from 'transformers' (C:\Users\Shyamal\anaconda3\lib\site-packages\transformers\__init__.py)

## Segment 2: Text Feature Engineering

**Explanation:** This extracts structured numerical features from the catalog text such as quantity, weight, and text complexity measures that are highly correlated with product pricing.

In [ ]:
import re

def extract_features_from_text(text):
    """Extract structured features from catalog content"""
    features = {}

    # Extract quantity/pack size
    quantity_patterns = [
        r'(\\d+)\\s*(?:Pack|pack|Count|count)',
        r'IPQ[:\\s]*(\\d+)',
        r'(\\d+)\\s*(?:Ounce|ounce|oz|Oz)',
        r'(\\d+\\.?\\d*)\\s*(?:Pound|pound|lb|Lb)'
    ]
    quantities = []
    for pattern in quantity_patterns:
        matches = re.findall(pattern, str(text))
        quantities.extend([float(m) for m in matches])
    features['quantity'] = max(quantities) if quantities else 1.0

    # Extract weight/volume
    weight_patterns = [
        r'(\\d+\\.?\\d*)\\s*(?:oz|ounce)',
        r'(\\d+\\.?\\d*)\\s*(?:lb|pound)',
        r'(\\d+\\.?\\d*)\\s*(?:ml|milliliter)',
        r'(\\d+\\.?\\d*)\\s*(?:l|liter)'
    ]
    weights = []
    for pattern in weight_patterns:
        matches = re.findall(pattern, str(text).lower())
        weights.extend([float(m) for m in matches])
    features['weight'] = max(weights) if weights else 0.0

    # Text length as complexity indicator
    features['text_length'] = len(str(text))
    features['word_count'] = len(str(text).split())

    return features

# Extract features from both datasets
print("Extracting text features...")
train_text_features = train_df['catalog_content'].apply(extract_features_from_text)
test_text_features = test_df['catalog_content'].apply(extract_features_from_text)

train_text_df = pd.DataFrame(train_text_features.tolist())
test_text_df = pd.DataFrame(test_text_features.tolist())

print(f"Extracted features: {train_text_df.columns.tolist()}")
print(train_text_df.head())

Extracting text features...
Extracted features: ['quantity', 'weight', 'text_length', 'word_count']
   quantity  weight  text_length  word_count
0       1.0     0.0           91          18
1       1.0     0.0          511          80
2       1.0     0.0          328          59
3       1.0     0.0         1318         211
4       1.0     0.0          155          28


## Segment 3: CLIP Model Setup for Multi-Modal Feature Extraction

**Explanation:** This creates an efficient CLIP-based feature extractor that processes images and text in batches for faster inference. CLIP produces normalized embeddings in a shared space where similar images and text have similar vectors. The batch processing significantly improves efficiency compared to single-sample processing.

In [ ]:
class CLIPFeatureExtractor:
    """Efficient CLIP-based feature extractor for images and text"""

    def __init__(self, model_name='openai/clip-vit-base-patch32', batch_size=32):
        """
        Initialize CLIP model
        Args:
            model_name: CLIP model variant (base is faster, large is more accurate)
            batch_size: Number of samples to process at once
        """
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"Using device: {self.device}")

        # Load CLIP model and processor
        self.model = CLIPModel.from_pretrained(model_name).to(self.device)
        self.processor = CLIPProcessor.from_pretrained(model_name)
        self.model.eval()
        self.batch_size = batch_size

        # Get embedding dimensions
        self.image_embed_dim = self.model.config.projection_dim
        self.text_embed_dim = self.model.config.projection_dim

        print(f"Model loaded: {model_name}")
        print(f"Embedding dimension: {self.image_embed_dim}")

    def get_image_embeddings(self, image_urls, max_retries=3):
        """Extract image embeddings using CLIP image encoder"""
        embeddings = []

        for i in tqdm(range(0, len(image_urls), self.batch_size), desc="Processing images"):
            batch_urls = image_urls[i:i+self.batch_size]
            batch_images = []

            # Download images
            for url in batch_urls:
                img = None
                for attempt in range(max_retries):
                    try:
                        response = requests.get(url, timeout=5)
                        img = Image.open(BytesIO(response.content)).convert('RGB')
                        break
                    except:
                        if attempt < max_retries - 1:
                            continue
                        else:
                            # Create blank image if download fails
                            img = Image.new('RGB', (224, 224), color='white')
                batch_images.append(img)

            # Process batch
            try:
                inputs = self.processor(images=batch_images, return_tensors="pt", padding=True)
                inputs = {k: v.to(self.device) for k, v in inputs.items()}

                with torch.no_grad():
                    image_features = self.model.get_image_features(**inputs)

                # Normalize embeddings
                image_features = image_features / image_features.norm(dim=-1, keepdim=True)
                embeddings.append(image_features.cpu().numpy())
            except Exception as e:
                print(f"Batch error: {e}")
                embeddings.append(np.zeros((len(batch_images), self.image_embed_dim)))

        return np.vstack(embeddings)

    def get_text_embeddings(self, texts):
        """Extract text embeddings using CLIP text encoder"""
        embeddings = []

        for i in tqdm(range(0, len(texts), self.batch_size), desc="Processing text"):
            batch_texts = texts[i:i+self.batch_size]

            # Truncate long texts
            batch_texts = [str(text)[:500] for text in batch_texts]

            try:
                inputs = self.processor(text=batch_texts, return_tensors="pt",
                                      padding=True, truncation=True)
                inputs = {k: v.to(self.device) for k, v in inputs.items()}

                with torch.no_grad():
                    text_features = self.model.get_text_features(**inputs)

                # Normalize embeddings
                text_features = text_features / text_features.norm(dim=-1, keepdim=True)
                embeddings.append(text_features.cpu().numpy())
            except Exception as e:
                print(f"Batch error: {e}")
                embeddings.append(np.zeros((len(batch_texts), self.text_embed_dim)))

        return np.vstack(embeddings)

# Initialize CLIP extractor
clip_extractor = CLIPFeatureExtractor(
    model_name='openai/clip-vit-base-patch32',  # Faster base model
    batch_size=32
)

Using device: cpu


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

## Segment 4: Extract CLIP Features

**Explanation:** This extracts semantic text embeddings from product descriptions using CLIP's text encoder. These embeddings capture the semantic meaning of the product descriptions in a 512-dimensional space. Then it extracts visual features from product images using CLIP's image encoder. CLIP's vision transformer processes images as sequences of patches to create rich visual representations.

In [ ]:
# Extract text embeddings from catalog content
print("\n=== Extracting CLIP Text Features ===")
train_text_embeddings = clip_extractor.get_text_embeddings(train_df['catalog_content'].tolist())
test_text_embeddings = clip_extractor.get_text_embeddings(test_df['catalog_content'].tolist())

print(f"Train text embeddings shape: {train_text_embeddings.shape}")
print(f"Test text embeddings shape: {test_text_embeddings.shape}")

# Save to avoid recomputation
np.save('train_clip_text_embeddings.npy', train_text_embeddings)
np.save('test_clip_text_embeddings.npy', test_text_embeddings)

In [ ]:
# Extract image embeddings
print("\n=== Extracting CLIP Image Features ===")
train_image_embeddings = clip_extractor.get_image_embeddings(train_df['image_link'].tolist())
test_image_embeddings = clip_extractor.get_image_embeddings(test_df['image_link'].tolist())

print(f"Train image embeddings shape: {train_image_embeddings.shape}")
print(f"Test image embeddings shape: {test_image_embeddings.shape}")

# Save to avoid recomputation
np.save('train_clip_image_embeddings.npy', train_image_embeddings)
np.save('test_clip_image_embeddings.npy', test_image_embeddings)

## Segment 5: Feature Combination and Dimensionality Reduction

**Explanation:** This combines CLIP's multi-modal embeddings with engineered features and applies dimensionality reduction for computational efficiency. SVD reduces feature dimensions while preserving most variance, speeding up training without significant accuracy loss.

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

# Combine all features
print("\n=== Combining Features ===")

# Concatenate CLIP embeddings with engineered features
X_train_combined = np.hstack([
    train_text_embeddings,  # CLIP text features
    train_image_embeddings,  # CLIP image features
    train_text_df.values  # Engineered numerical features
])

X_test_combined = np.hstack([
    test_text_embeddings,
    test_image_embeddings,
    test_text_df.values
])

print(f"Combined feature shape: {X_train_combined.shape}")

# Optional: Apply dimensionality reduction for efficiency
use_svd = True
n_components = 256

if use_svd and X_train_combined.shape[1] > n_components:
    print(f"\nApplying SVD to reduce dimensions from {X_train_combined.shape[1]} to {n_components}")
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    X_train_reduced = svd.fit_transform(X_train_combined)
    X_test_reduced = svd.transform(X_test_combined)

    print(f"Explained variance ratio: {svd.explained_variance_ratio_.sum():.4f}")
else:
    X_train_reduced = X_train_combined
    X_test_reduced = X_test_combined

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_reduced)
X_test_scaled = scaler.transform(X_test_reduced)

# Target variable
y_train = train_df['price'].values

print(f"\nFinal training features shape: {X_train_scaled.shape}")
print(f"Final test features shape: {X_test_scaled.shape}")

## Segment 6: Model Training with LightGBM

**Explanation:** This trains a LightGBM model using k-fold cross-validation for robust performance estimation. LightGBM is chosen for its efficiency with large feature sets and ability to handle the combined multi-modal features effectively. Cross-validation provides reliable performance metrics and reduces overfitting.

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_percentage_error

def smape(y_true, y_pred):
    """Calculate Symmetric Mean Absolute Percentage Error"""
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_true - y_pred) / denominator
    return np.mean(diff) * 100

# LightGBM parameters optimized for efficiency and accuracy
lgb_params = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': 8,
    'min_child_samples': 20,
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

# Cross-validation for robust training
n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

oof_predictions = np.zeros(len(X_train_scaled))
test_predictions = np.zeros(len(X_test_scaled))
fold_scores = []

print("\n=== Training LightGBM with Cross-Validation ===")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_scaled), 1):
    print(f"\nFold {fold} / {n_folds}")

    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]

    # Create LightGBM datasets
    train_data = lgb.Dataset(X_fold_train, label=y_fold_train)
    val_data = lgb.Dataset(X_fold_val, label=y_fold_val, reference=train_data)

    # Train model
    model = lgb.train(
        lgb_params,
        train_data,
        num_boost_round=1000,
        valid_sets=[train_data, val_data],
        valid_names=['train', 'valid'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=100)
        ]
    )

    # Predictions
    oof_predictions[val_idx] = model.predict(X_fold_val)
    test_predictions += model.predict(X_test_scaled) / n_folds

    # Calculate SMAPE
    fold_smape = smape(y_fold_val, oof_predictions[val_idx])
    fold_scores.append(fold_smape)
    print(f"Fold {fold} SMAPE: {fold_smape:.4f}%")

# Overall validation score
overall_smape = smape(y_train, oof_predictions)
print(f"\n{'='*50}")
print(f"Overall Out-of-Fold SMAPE: {overall_smape:.4f}%")
print(f"Mean Fold SMAPE: {np.mean(fold_scores):.4f}% (+/- {np.std(fold_scores):.4f}%)")
print(f"{'='*50}")

## Segment 7: Post-Processing and Submission Generation

**Explanation:** This generates the final submission file in the required format, ensuring all predictions are positive values and no samples are missing.

In [ ]:
# Ensure predictions are positive
test_predictions = np.maximum(test_predictions, 0.01)

# Create submission file
submission_df = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': test_predictions
})

# Save submission
submission_df.to_csv('test_out.csv', index=False)

print("\n=== Submission Statistics ===")
print(submission_df['price'].describe())
print(f"\nSubmission file saved: test_out.csv")
print(f"Total predictions: {len(submission_df)}")

# Verify output format
print("\n=== Sample Predictions ===")
print(submission_df.head(10))

# Check for any issues
if submission_df['price'].isna().any():
    print("\nWARNING: NaN values detected in predictions!")
if (submission_df['price'] <= 0).any():
    print("\nWARNING: Non-positive prices detected!")
else:
    print("\n✓ All predictions are positive")
    print("✓ No missing values")
    print("✓ Submission ready for upload")

## Segment 8: Model Analysis and Feature Importance

**Explanation:** This analyzes which features contribute most to price prediction, providing insights into what drives product pricing in the model.

In [ ]:
# Feature importance analysis
print("\n=== Feature Importance Analysis ===")

# Train final model on full data for feature importance
final_model = lgb.train(
    lgb_params,
    lgb.Dataset(X_train_scaled, label=y_train),
    num_boost_round=500,
    verbose_eval=False
)

# Get feature importance
importance = final_model.feature_importance(importance_type='gain')
feature_names = [f'Feature_{i}' for i in range(X_train_scaled.shape[1])]

# Sort and display top features
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features:")
print(importance_df.head(20))

# Save model for future use
final_model.save_model('clip_lgb_price_model.txt')
print("\n✓ Model saved: clip_lgb_price_model.txt")
